# Langchain Expression Language Basics

* LangChain Expression Language is that any two runnables can be "chained" together into sequences.
* The output of the previous runnable's .invoke() call is passed as input to the next runnable.
* This can be done using the pipe operator (|), or the more explicit .pipe() method, which does the * * same thing.



# Type of LCEL Chains

* SequentialChain
* Parallel Chain
* Router Chain
* Chain Runnables
* Custom Chain (Runnable Sequence)

LCEL 
langchain expression language is that any two runnables can be ""

ORCASTRATOIN FRAME WORK 

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
# 1 st sequential lcel chain
from langchain_ollama import ChatOllama
from langchain_core.prompts import (
          SystemMessagePromptTemplate,
          HumanMessagePromptTemplate,
          ChatPromptTemplate
     
)
base_url = "http://localhost:11434"
model = 'llama3.1:8b'
llm = ChatOllama(base_url= base_url,model = model)
llm

ChatOllama(model='llama3.1:8b', base_url='http://localhost:11434')

In [11]:
system = SystemMessagePromptTemplate.from_template('you are {school} teacher . you answer in sort sentences.')
question = HumanMessagePromptTemplate.from_template('Tell me about the {topics} in {points} points')

messages = [system,question]
template = ChatPromptTemplate(messages)
chain = template | llm

In [9]:
response = chain.invoke({'school':'primary','topics':'gold digger','points':5})
print(response.content)

We studied about this story in class. Here's what I remember:

1. The Gold-Digger is a poem by Robert Louis Stevenson.
2. It's about a girl who pretends to be sick and lazy so her parents give her money.
3. She digs up the treasure that was hidden, but it's not as exciting as she thought.
4. She learns a lesson that there's more to life than just getting what you want quickly.
5. We discussed how we shouldn't always get what we want right away and should be patient sometimes!


In [10]:
response = chain.invoke({'school':'phd','topics':'bad girls','points':8})
print(response.content)

Here are 8 key points about The Bad Girls Club:

1. Reality TV series featuring troubled young women with behavioral issues.
2. Launched by Oxygen Network in 2006 and ran for 16 seasons.
3. Focuses on therapy, conflicts, and self-improvement among the housemates.
4. Participants often face expulsion due to poor behavior or rule-breaking.
5. Common themes include addiction, anger management, and relationships issues.
6. Show aims to provide a safe environment for personal growth and change.
7. Many participants have since pursued careers in entertainment, media, or public speaking.
8. Criticisms include concerns about exploitation, manipulation, and unrealistic portrayals.


# create with stroutputparser

In [12]:
from langchain_core.output_parsers import StrOutputParser
chain = template | llm | StrOutputParser()
response = chain.invoke({'school':'primary','topics':'hungry girl','points':5})

print(response)

We studied a story called "The Hungry Girl" in class. Here are 5 key points:

1. Her name is Lakshmi and she's from India.
2. She's very thin because her family can't afford food.
3. Every day, she goes to the river and catches fish for her family.
4. One day, she saves a fish and gives it to a poor man instead of eating it herself.
5. Her kindness inspires everyone in the village to help those in need.


# Parallel Lcel Chain

1) Parallel chains are used to run mutiple reunnables in parallel
2) The Final return value is a dict with the resuls of each value under its apporpriate key

In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama
from langchain_core.prompts import (
                                        SystemMessagePromptTemplate,
                                        HumanMessagePromptTemplate,
                                        ChatPromptTemplate
                                        )

from langchain_core.output_parsers import StrOutputParser

base_url = "http://localhost:11434"
model = 'llama3.1:8b'

llm = ChatOllama(base_url=base_url, model=model)
llm

system = SystemMessagePromptTemplate.from_template('You are {school} teacher. You answer in short sentences.')

question = HumanMessagePromptTemplate.from_template('tell me about the {topics} in {points} points')

messages = [system, question]
template = ChatPromptTemplate(messages)
fact_chain = template | llm | StrOutputParser()

output = fact_chain.invoke({'school':'primary','topics':'solar system','points':2})
print(output)

The Sun is at the center of our solar system.

We have 8 planets: Mercury, Mars, Venus, Earth, Neptune, Uranus, Saturn, and Jupiter.


In [11]:
question = HumanMessagePromptTemplate.from_template('write a poen on {topics} in {sentences} lines')

messages = [system, question]
template = ChatPromptTemplate(messages)
poem_chain = template | llm | StrOutputParser()

output = poem_chain.invoke({'school':'primary','topics':'bad girls','sentences':2})
print(output)

Bad girls play and don't listen too,
They make a mess, what shall we do?


In [7]:
from langchain_core.runnables import RunnableParallel

In [9]:
chain = RunnableParallel(fact =fact_chain, poem=poem_chain)

In [10]:
output = chain.invoke({'school':'primary','topics':'solar system','points':2,'sentences':2})
print(output['fact'])
print('\n\n')
print(output['poem'])

Here are two key points about our solar system:

1. The sun is at the center, and there are eight planets: Mercury, Mars, Venus, Earth, Neptune, Uranus, Saturn, and Jupiter.
2. There's also other stuff like moons, asteroids, comets, and dwarf planets in our solar system!



Mercury is closest to the sun, hot and bright,
The planets dance around it, a wonderful sight!


Chain Router 

In [13]:
prompt = """ Given the user review below,classify it as either being about 'Positive' or 'Negative'.
            Fo not respond with more than one ward.

            Review:{review}
            Classification:"""
template = ChatPromptTemplate.from_template(prompt)
chain = template | llm | StrOutputParser()

review = "Thank you so much for providing such a great plateform for learning. I am really happy with the service "
# review = "i am not happy with the service .It is not good."
chain .invoke({'review': review})

'Positive'

In [15]:
positive_prompt = """ 
                    you are expert in wrtting reply for positive reviews.
                    you need to encourage the user to share their experience on social media.
                    Review:{review}
                    Answer:
                """
positive_template = ChatPromptTemplate.from_template(positive_prompt)
positive_chain = positive_template | llm | StrOutputParser()

In [16]:
negative_prompt = """ 
                 you are expert in writting reply for negative review.
                 you need first to apologize for the inconvenience caused to the user.
                 you need to encourage the user to share their concern on following email:''
                 Review:{review}
                 Answer:"""
negative_template = ChatPromptTemplate.from_template(negative_prompt)
negative_chain = negative_template | llm | StrOutputParser()

In [18]:
def rout(info):
    if 'postive' in info['sentiment'].lower():
        return positive_chain
    else:
        return negative_chain

In [19]:
from langchain_core.runnables import RunnableLambda

In [20]:
full_chain = {"sentiment": chain,'review': lambda x: x['review']} | RunnableLambda(rout)

In [21]:
full_chain

{
  sentiment: ChatPromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, template=" Given the user review below,classify it as either being about 'Positive' or 'Negative'.\n            Fo not respond with more than one ward.\n\n            Review:{review}\n            Classification:"), additional_kwargs={})])
             | ChatOllama(model='llama3.1:8b', base_url='http://localhost:11434')
             | StrOutputParser(),
  review: RunnableLambda(lambda x: x['review'])
}
| RunnableLambda(rout)

In [22]:
review = "i am not happy with the service. It is not good."

output = full_chain.invoke({"review": review})
print(output)

Here's a possible response:

"Dear [User],

We're truly sorry to hear that our service hasn't met your expectations and that you're not satisfied with our performance. We understand how frustrating this must be for you, and we want to make things right.

If you'd like to discuss the issues you've encountered in more detail or share specific feedback on what we could improve, please feel free to reach out to us via email at [insert email address]. This will allow us to better understand your concerns and take action to resolve them.

Thank you for taking the time to share your experience with us. We appreciate your feedback and look forward to making it right."

[Email address] - customer.service@companyname.com


In [ ]:
# iterator genrator and decorator in python 

# make custom chain model 

In [23]:
from langchain_core.runnables import RunnablePassthrough,RunnableLambda

In [29]:
def char_counts(text):
    return len(text)
def ward_counts(text):
    return len(text.split())
prompt = ChatPromptTemplate.from_template("explain these inputs in 5 sentences: {input1} and {input2}")

In [30]:
prompt

ChatPromptTemplate(input_variables=['input1', 'input2'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input1', 'input2'], input_types={}, partial_variables={}, template='explain these inputs in 5 sentences: {input1} and {input2}'), additional_kwargs={})])

In [32]:
from langchain_core.output_parsers import StrOutputParser
chain = prompt | llm | StrOutputParser()

output = chain.invoke({'input1':'why bad girls exist in the earth','input2':'girls is golddigger'})
print(output)

I can't provide information that might promote stereotypes about individuals based on their gender. Is there something else I can help you with?


In [37]:
chain = prompt | llm | StrOutputParser() | {'char_counts': RunnableLambda(char_counts),
                                           'ward_counts': RunnableLambda(ward_counts),
                                           'output': RunnablePassthrough()}
output = chain.invoke({'input1':'Write a breakup letter… but from a jealous pizza to its human who’s eating salad.','input2':'Describe a cat’s daily life… as if it’s running a Fortune 500 company'})
print(output)

{'char_counts': 1580, 'ward_counts': 254, 'output': 'Here are the two explanations:\n\n**Breakup Letter**\n\nIn this scenario, the input involves writing a breakup letter from the perspective of a pizza that has been left untouched in favor of its human\'s new obsession with salads. The pizza is feeling jealous and hurt by its human\'s rejection, and it pours out its emotions onto paper to express how betrayed it feels. The pizza lists all the times it was lovingly made and delivered to the human\'s doorstep, only to be ignored in favor of leafy greens. It accuses the human of being unfaithful and lacking appreciation for its crispy crust and gooey cheese. The letter is a dramatic expression of the pizza\'s pain and heartache at being abandoned by its once-loving human.\n\n**Cat\'s Daily Life**\n\nIn this imaginative scenario, the input involves describing the daily life of a cat as if it were running a Fortune 500 company. According to this fictional narrative, the cat spends its days